# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from matplotlib import pyplot as plt
import pandas as pd
from IPython.core.display_functions import clear_output

# Load data

In [ ]:
data_folder = "../../data/preprocessed-data"
df_eyewire2_field_level = pd.read_hdf(os.path.join(data_folder, 'df_fields.h5'), key='dataframe')

In [ ]:
df_eyewire2_roi_level = pd.concat([pd.read_hdf(os.path.join(data_folder, f'df_eyewire2_roi_level_GCL{i}.h5'), key='dataframe') for i in range(5)])
df_eyewire2_roi_level['date'] = df_eyewire2_roi_level['date'].astype(str)
df_eyewire2_roi_level['exp_num'] = 2
df_eyewire2_roi_level['roi_id'] = df_eyewire2_roi_level['roi_id'].astype(int)
df_eyewire2_roi_level = df_eyewire2_roi_level.set_index(['date', 'exp_num', 'field', 'roi_id'])
df_eyewire2_roi_level.head()

In [ ]:
morph_folder = "../../data/morphological-data"
morph_spreadsheet_filename = "Eyewire II Proofread Cells Master List - All Cells (WIP) 2025-09-03.csv"

morph_df = pd.read_csv(os.path.join(os.path.join(morph_folder, morph_spreadsheet_filename)), dtype=str)
print(morph_df.shape)
morph_df.head()

In [ ]:
mapping_spreadsheet_filename = "roi_mapping_2p_to_em.csv"

mapping_df = pd.read_csv(os.path.join(os.path.join(morph_folder, mapping_spreadsheet_filename)), dtype=str)
mapping_df['exp_num'] = 2
mapping_df['roi_id'] = mapping_df['roi_id'].astype(int)
mapping_df = mapping_df.set_index(['date', 'exp_num', 'field', 'roi_id'])
print(mapping_df.shape)
mapping_df.head()

In [ ]:
nuc_col = 'Updated Nuc ID (Sept 2)'

assert nuc_col in morph_df.columns, f"Column '{nuc_col}' not found in morph_df"
assert nuc_col in mapping_df.columns, f"Column '{nuc_col}' not found in mapping_df"

df = df_eyewire2_roi_level.join(mapping_df)
df = df.reset_index().set_index(nuc_col)

morph_df_ = morph_df[morph_df[nuc_col].notna()].set_index(nuc_col)
morph_df_ = morph_df_[~morph_df_.index.duplicated(keep='first')]
df = pd.merge(df, morph_df_, left_index=True, right_index=True, how='inner').reset_index()
df['qfilt'] = (df['bar_qidx'] > 0.6) | (df['chirp_qidx'] > 0.45)

print(list(df.columns))
print(df.shape)
df.head()

# Plot data

In [ ]:
from eyewire2_functional_analysis.plot_dataframe import plot_df_chirp_and_bar

In [ ]:
fig_dir = './figures/responses_by_field'
os.makedirs(fig_dir, exist_ok=True)

for (field, cell_class), df_plot in df.groupby(['field', 'Cell Class']):
    clear_output()
    fig, axs = plot_df_chirp_and_bar(df=df_plot, id_col=nuc_col, type_col='Cell Type',
                                     title=f"{field}: All {cell_class}s", chirp_q_thresh=0.45, bar_q_thresh=0.6)
    plt.savefig(os.path.join(fig_dir, f'response_overview_{field}_{cell_class}.pdf'))
    plt.close(fig)

In [ ]:
fig_dir = './figures/responses_by_type'
os.makedirs(fig_dir, exist_ok=True)

def simplify_cell_type(cell_type_):
    # Return None if the cell type is None or NaN
    if cell_type_ is None or pd.isna(cell_type_):
        return None

    # Convert to string in case it's not already
    cell_type_ = str(cell_type_)

    # Remove question marks
    cell_type_ = cell_type_.replace('?', '')

    # Remove expressions of uncertainty
    uncertainty_phrases = ['maybe', 'I think', 'probably', 'or', '-ish']
    for phrase in uncertainty_phrases:
        # Split by phrase and take the first part
        if phrase in cell_type_.lower():
            parts = cell_type_.lower().split(phrase)
            cell_type_ = parts[0].strip()

    # If the cell type became empty after cleaning, return None
    if not cell_type_ or cell_type_.isspace():
        return None

    # Return the cleaned cell type
    return cell_type_.strip()

df['Cell Type simplified'] = df['Cell Type'].apply(simplify_cell_type)

for cell_type, df_plot in df.groupby('Cell Type simplified'):
    clear_output()
    fig, axs = plot_df_chirp_and_bar(
        df=df_plot.sort_values(['field', 'roi_id'], inplace=False),
        id_col=nuc_col, type_col='Cell Type',
        title=cell_type, chirp_q_thresh=0.45, bar_q_thresh=0.6)
    plt.savefig(os.path.join(fig_dir, f'response_overview_{cell_type}.pdf'))
    plt.close(fig)